# SmarTRIZ DPO Sandbox Dry-Run

**Purpose:** Validate the nb02 DPO pair-matching logic and nb03 eval metric helpers against real local data — without touching the main Colab notebooks or writing any files to `data/`.

**Read-only:** This notebook never writes to `data/`, `src/`, or any other notebook.

**Run order:** Run All (Kernel → Restart & Run All). Expected runtime: ~20-40 seconds.

---

## Bölüm 1 — DPO Pair Matching

Reproduces `notebooks/02_dpo_training.ipynb` cell-5 verbatim with local paths.

In [ ]:
# ── Path setup (local repo, no Google Drive) ──────────────────────────────
from pathlib import Path

# notebooks/sandbox/ → repo root → data/
REPO_ROOT  = Path('__file__').resolve().parents[2] if '__file__' in dir() else Path.cwd()
# Handle running from notebooks/sandbox/ or from repo root
for candidate in [
    Path.cwd() / 'data',
    Path.cwd().parent / 'data',
    Path.cwd().parents[1] / 'data',
]:
    if (candidate / 'judged.jsonl').exists():
        DATA_DIR = candidate
        break
else:
    raise FileNotFoundError('Cannot locate data/ directory. Run notebook from repo root or notebooks/sandbox/')

REJECTED_PATH = DATA_DIR / 'rejected_dataset.jsonl'
JUDGED_PATH   = DATA_DIR / 'judged.jsonl'

print(f'DATA_DIR:      {DATA_DIR}')
print(f'rejected path: {REJECTED_PATH}  exists={REJECTED_PATH.exists()}')
print(f'judged path:   {JUDGED_PATH}  exists={JUDGED_PATH.exists()}')

In [ ]:
# ── Load raw data ─────────────────────────────────────────────────────────
import json
from collections import Counter

def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

rejected_raw = load_jsonl(REJECTED_PATH)
judged_all   = load_jsonl(JUDGED_PATH)

print(f'rejected_dataset.jsonl: {len(rejected_raw):,} records')
print(f'judged.jsonl:           {len(judged_all):,} records')

stage_counts = Counter(r.get('stage', '?') for r in rejected_raw)
print('\nrejected stage breakdown:')
for stage, count in sorted(stage_counts.items(), key=lambda x: -x[1]):
    print(f'  {stage:<15} {count:>6}')

In [ ]:
# ── Build chosen indexes from judged PASS records ─────────────────────────
# (verbatim from nb02 cell-5)

judged_pass = [
    j for j in judged_all
    if j.get('meta', {}).get('judge_scores', {}).get('verdict') == 'PASS'
]
judged_pass_by_id = {j['id']: j for j in judged_pass}

judged_pass_by_parent = {}
for j in judged_pass:
    pid = j.get('meta', {}).get('parent_seed_id')
    if pid:
        judged_pass_by_parent.setdefault(pid, []).append(j)

def best_chosen_for_parent(pid):
    candidates = judged_pass_by_parent.get(pid, [])
    if not candidates:
        return None
    return max(
        candidates,
        key=lambda j: j.get('meta', {}).get('judge_scores', {}).get('average', 0.0)
    )

def format_assistant_from_judged(j):
    cp = j.get('contradiction_pair', {})
    principles = ', '.join(j.get('inventive_principles', []))
    return (
        f'<think>\n{j.get("reasoning_chain", "")}\n</think>\n'
        f'Contradiction:\n\n'
        f'Improving: {cp.get("improving_parameter", "")}\n'
        f'Worsening: {cp.get("worsening_parameter", "")}\n\n'
        f'Inventive Principles: {principles}\n'
        f'Solution:\n{j.get("solution", "")}'
    )

def format_assistant_from_case(case):
    cp = case.get('contradiction_pair', {})
    principles = ', '.join(case.get('inventive_principles', []))
    return (
        f'<think>\n{case.get("reasoning_chain", "")}\n</think>\n'
        f'Contradiction:\n\n'
        f'Improving: {cp.get("improving_parameter", "")}\n'
        f'Worsening: {cp.get("worsening_parameter", "")}\n\n'
        f'Inventive Principles: {principles}\n'
        f'Solution:\n{case.get("solution", "")}'
    )

print(f'judged PASS records:         {len(judged_pass):,}')
print(f'judged_pass_by_id size:      {len(judged_pass_by_id):,}')
print(f'judged_pass_by_parent size:  {len(judged_pass_by_parent):,} unique parent_seed_ids')

# Verdict distribution
verdict_counts = Counter(
    j.get('meta', {}).get('judge_scores', {}).get('verdict', 'NO_VERDICT')
    for j in judged_all
)
print(f'\njudged verdict breakdown: {dict(verdict_counts)}')

In [ ]:
# ── Identify usable rejected records ──────────────────────────────────────
# (verbatim from nb02 cell-5)

EXCLUDE_STAGES = {'copy_sweep', 'matrix'}
matrix_case_ids = {
    r['case']['id']
    for r in rejected_raw
    if r.get('stage') == 'matrix' and r.get('case', {}).get('id')
}

candidate_rejected = [
    r for r in rejected_raw
    if r.get('stage') not in EXCLUDE_STAGES
    and r.get('case', {}).get('problem')
    and r.get('case', {}).get('solution')
    and r.get('case', {}).get('reasoning_chain')
    and r.get('case', {}).get('id') not in matrix_case_ids
    and r.get('case', {}).get('id') not in judged_pass_by_id  # skip cases that passed judgment (same content → identical pair)
]

judged_fail = [
    j for j in judged_all
    if j.get('meta', {}).get('judge_scores', {}).get('verdict') == 'FAIL'
    and j.get('problem') and j.get('solution') and j.get('reasoning_chain')
]

candidate_stage_breakdown = Counter(r.get('stage') for r in candidate_rejected)

print(f'matrix_case_ids excluded:    {len(matrix_case_ids):,}')
print(f'Usable rejected candidates:  {len(candidate_rejected):,}')
print(f'  Stage breakdown: {dict(candidate_stage_breakdown)}')
print(f'Judged FAIL candidates:      {len(judged_fail):,}')

In [ ]:
# ── Pass 1: match rejected → chosen via cascade id → parent_seed_id ───────
# (verbatim from nb02 cell-5)

dpo_pairs = []
unmatched = []

match_by_id     = 0
match_by_parent = 0

for rej in candidate_rejected:
    case    = rej['case']
    case_id = case.get('id', '')
    pid     = case.get('meta', {}).get('parent_seed_id')

    chosen_j = judged_pass_by_id.get(case_id)
    if chosen_j is not None:
        match_by_id += 1
    elif pid:
        chosen_j = best_chosen_for_parent(pid)
        if chosen_j is not None:
            match_by_parent += 1

    if chosen_j is None:
        unmatched.append({'source': 'rejected_dataset', 'record': rej})
        continue

    dpo_pairs.append({
        'prompt':   case.get('problem', ''),
        'chosen':   format_assistant_from_judged(chosen_j),
        'rejected': format_assistant_from_case(case),
        '_source':  rej.get('stage', '?'),
    })

print(f'Pass 1 results:')
print(f'  Matched by case.id:         {match_by_id}')
print(f'  Matched by parent_seed_id:  {match_by_parent}')
print(f'  Unmatched:                  {len(unmatched)}')
print(f'  Pairs after Pass 1:         {len(dpo_pairs)}')

In [ ]:
# ── Pass 2: judged FAIL as rejected, best PASS sibling as chosen ──────────
# (verbatim from nb02 cell-5)

seen_fail_ids   = set()
pass2_matched   = 0
pass2_unmatched = 0

for jf in judged_fail:
    jf_id = jf.get('id', '')
    if jf_id in seen_fail_ids:
        continue
    pid      = jf.get('meta', {}).get('parent_seed_id')
    chosen_j = best_chosen_for_parent(pid) if pid else None
    if chosen_j is None or chosen_j.get('id') == jf_id:
        unmatched.append({'source': 'judged_fail', 'record': jf})
        pass2_unmatched += 1
        continue
    dpo_pairs.append({
        'prompt':   jf.get('problem', ''),
        'chosen':   format_assistant_from_judged(chosen_j),
        'rejected': format_assistant_from_judged(jf),
        '_source':  'judged_fail',
    })
    seen_fail_ids.add(jf_id)
    pass2_matched += 1

print(f'Pass 2 results:')
print(f'  judged FAIL candidates:     {len(judged_fail)}')
print(f'  Matched (FAIL→PASS sibling):{pass2_matched}')
print(f'  Unmatched:                  {pass2_unmatched}')
print(f'  Pairs after Pass 2:         {len(dpo_pairs)}')

In [ ]:
# ── Deduplication + validation ────────────────────────────────────────────
# (verbatim from nb02 cell-5, with soft warning instead of hard assert for dry-run)

seen_prompts = set()
deduped = []
for p in dpo_pairs:
    key = p['prompt'][:120]
    if key not in seen_prompts:
        seen_prompts.add(key)
        deduped.append(p)
dpo_pairs = deduped

assert len(dpo_pairs) > 0, 'DPO matching produced 0 pairs — check judged.jsonl path'

# Defense layer: mirrors nb02 cell-5 fix — should filter 0 pairs if smart filter works
before_filter = len(dpo_pairs)
dpo_pairs = [p for p in dpo_pairs if p['chosen'] != p['rejected']]
filtered_identical = before_filter - len(dpo_pairs)
IDENTICAL_BUG_FOUND = filtered_identical > 0

if IDENTICAL_BUG_FOUND:
    print(f'WARNING: {filtered_identical} identical chosen/rejected pairs slipped through smart filter.')
    print('  Smart filter (judged_pass_by_id exclusion) may have missed some cases.')
    print('  >>> INVESTIGATE: nb02 cell-5 smart filter may need adjustment <<<')
else:
    print(f'Smart filter OK: 0 identical pairs (defense layer filtered nothing).')

source_breakdown = Counter(p.get('_source', '?') for p in dpo_pairs)

print(f'\nAfter deduplication + identity filter: {len(dpo_pairs)} valid DPO pairs')
print(f'Source breakdown: {dict(source_breakdown)}')
print(f'Total unmatched (need teacher if < 1500): {len(unmatched)}')

In [ ]:
# ── Sample pair preview (3 examples) ──────────────────────────────────────

print('=' * 70)
for i, pair in enumerate(dpo_pairs[:3]):
    src = pair.get('_source', '?')
    print(f'\n--- Pair {i+1}  [source: {src}] ---')
    print(f'PROMPT (first 150 chars):\n  {pair["prompt"][:150]}')
    print(f'CHOSEN (first 200 chars):\n  {pair["chosen"][:200]}')
    print(f'REJECTED (first 200 chars):\n  {pair["rejected"][:200]}')
    print()

### Bölüm 1 Görselleştirme — Pair Kaynak Dağılımı

In [ ]:
# ── Plot 1: pair source bar chart ─────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

source_counts = Counter(p.get('_source', 'unknown') for p in dpo_pairs)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('DPO Pair Dry-Run: Source & Chosen Quality', fontsize=13)

# Left: source bar
ax = axes[0]
labels = list(source_counts.keys())
values = list(source_counts.values())
bars = ax.bar(labels, values, color=['#4c72b0', '#dd8452', '#55a868', '#c44e52'][:len(labels)])
ax.set_title('Pair count by source')
ax.set_xlabel('Source')
ax.set_ylabel('Count')
ax.axhline(y=1500, color='red', linestyle='--', alpha=0.6, label='teacher threshold (1500)')
ax.axhline(y=200,  color='orange', linestyle='--', alpha=0.6, label='min viable (200)')
ax.legend(fontsize=8)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, str(val),
            ha='center', va='bottom', fontsize=9)

# Right: chosen judge average score histogram
ax2 = axes[1]
chosen_scores = []
for pair in dpo_pairs:
    # Find the chosen record's judge average (pair was built from judged_pass)
    # Use the pass records we already loaded; match by first 120 chars of prompt
    pass  # scores gathered below

# Collect judge average from judged_pass via prompt key
prompt_to_score = {
    j.get('problem', '')[:120]: j.get('meta', {}).get('judge_scores', {}).get('average')
    for j in judged_pass
    if j.get('meta', {}).get('judge_scores', {}).get('average') is not None
}
for pair in dpo_pairs:
    score = prompt_to_score.get(pair['prompt'][:120])
    if score is not None:
        chosen_scores.append(float(score))

if chosen_scores:
    ax2.hist(chosen_scores, bins=15, color='#4c72b0', edgecolor='white', alpha=0.85)
    ax2.set_title(f'Chosen judge average scores (n={len(chosen_scores)})')
    ax2.set_xlabel('Judge average score')
    ax2.set_ylabel('Count')
    ax2.axvline(x=7.0, color='red', linestyle='--', alpha=0.7, label='min threshold (7.0)')
    ax2.legend(fontsize=8)
else:
    ax2.text(0.5, 0.5, 'No score data available\n(judged PASS records may lack judge_scores)',
             ha='center', va='center', transform=ax2.transAxes, fontsize=10, color='gray')
    ax2.set_title('Chosen judge average scores')

plt.tight_layout()
plt.show()

print(f'Total pairs plotted: {len(dpo_pairs)}')
if chosen_scores:
    import statistics
    print(f'Chosen score stats: min={min(chosen_scores):.2f}  median={statistics.median(chosen_scores):.2f}  max={max(chosen_scores):.2f}')

---

## Bölüm 2 — Eval Metric Helpers

Reproduces the metric helper functions from `notebooks/03_convert_and_eval.ipynb` cell-6, then runs inline assertions.

In [ ]:
# ── Metric helpers (verbatim from nb03 cell-6, minus llm_judge + solution_cosine) ──
import re

_PRINCIPLE_NAMES = {
    1: ['segmentation', 'segmented'],
    2: ['taking out', 'extraction'],
    3: ['local quality'],
    4: ['asymmetry'],
    5: ['merging', 'consolidation'],
    6: ['universality'],
    7: ['nested doll', 'nesting'],
    8: ['anti-weight', 'counterweight'],
    9: ['preliminary anti-action', 'prior counteraction'],
    10: ['preliminary action', 'prior action'],
    11: ['beforehand cushioning', 'beforehand compensation'],
    12: ['equipotentiality'],
    13: ['the other way round', 'inversion'],
    14: ['spheroidality', 'curvature'],
    15: ['dynamics', 'dynamism'],
    16: ['partial or excessive actions'],
    17: ['another dimension', 'moving to another dimension'],
    18: ['mechanical vibration'],
    19: ['periodic action'],
    20: ['continuity of useful action'],
    21: ['skipping', 'rushing through'],
    22: ['blessing in disguise', 'convert harm to benefit'],
    23: ['feedback'],
    24: ['intermediary', 'mediator'],
    25: ['self-service'],
    26: ['copying'],
    27: ['cheap short-living', 'cheap disposable'],
    28: ['mechanics substitution', 'replacement of mechanical system'],
    29: ['pneumatics and hydraulics', 'pneumatics', 'hydraulics'],
    30: ['flexible shells', 'thin films'],
    31: ['porous materials'],
    32: ['color changes', 'optical properties'],
    33: ['homogeneity'],
    34: ['discarding and recovering', 'rejecting and regenerating'],
    35: ['parameter changes', 'transformation of properties'],
    36: ['phase transitions', 'phase transition'],
    37: ['thermal expansion'],
    38: ['strong oxidants', 'enriched atmosphere'],
    39: ['inert atmosphere', 'inert environment'],
    40: ['composite materials', 'composites'],
}

def extract_principle_numbers(text):
    found = set()
    t = text.lower()
    for m in re.finditer(r'#\s*(\d{1,2})', t):
        n = int(m.group(1))
        if 1 <= n <= 40:
            found.add(n)
    for m in re.finditer(r'\bprinciple\s+(\d{1,2})\b', t):
        n = int(m.group(1))
        if 1 <= n <= 40:
            found.add(n)
    for m in re.finditer(r'\b(\d{1,2})-[A-Za-z]', text):
        n = int(m.group(1))
        if 1 <= n <= 40:
            found.add(n)
    for num, aliases in _PRINCIPLE_NAMES.items():
        if any(alias in t for alias in aliases):
            found.add(num)
    return found


def principle_accuracy(generated, reference_principles):
    ref_nums = extract_principle_numbers(' '.join(reference_principles))
    gen_nums = extract_principle_numbers(generated)
    return len(ref_nums & gen_nums) >= min(2, len(ref_nums))


def contradiction_accuracy(generated, ref_cp):
    gen_lower = generated.lower()

    def _check_param(param_str):
        id_match = re.search(r'\(#(\d{1,2})\)', param_str)
        if id_match:
            n = id_match.group(1)
            return f'(#{n})' in generated or bool(re.search(rf'\b{n}\b', generated))
        content_words = [
            w for w in param_str.lower().split()
            if len(w) >= 5 and not re.match(r'[#()\d]+', w)
        ]
        if not content_words:
            return False
        matched = sum(1 for w in content_words if w in gen_lower)
        return matched >= max(1, int(len(content_words) * 0.6))

    return _check_param(ref_cp.get('improving_parameter', '')) and \
           _check_param(ref_cp.get('worsening_parameter', ''))


print('Metric helpers defined')

In [ ]:
# ── Assertion suite — extract_principle_numbers ───────────────────────────
# Plan verification cases (lines 88-90 of 2026-05-07-notebook-fixes.md)
failures = []

def chk(condition, label):
    if condition:
        print(f'  PASS  {label}')
    else:
        print(f'  FAIL  {label}')
        failures.append(label)

print('--- extract_principle_numbers ---')

# From plan
chk(35 in extract_principle_numbers('Use Principle 35 for parameter changes'),
    '#35 via "Principle N" format')
chk(40 in extract_principle_numbers('Apply #40 composite material approach'),
    '#40 via "#N" format')
chk(1 in extract_principle_numbers('segmentation technique applied'),
    '#1 via named alias "segmentation"')

# Additional alias tests
chk(40 in extract_principle_numbers('We use composites to solve this'),
    '#40 via alias "composites"')
chk(5 in extract_principle_numbers('Merging components reduces cost'),
    '#5 via alias "merging"')
chk(23 in extract_principle_numbers('Add a feedback loop to stabilize'),
    '#23 via alias "feedback"')

# Hyphenated format (10-Name)
chk(10 in extract_principle_numbers('Apply 10-preliminary action before welding'),
    '#10 via "N-Name" hyphenated format')

# Boundary: 0 and 41 should NOT be caught
chk(0  not in extract_principle_numbers('#0 is not a valid principle'),
    '#0 out-of-range correctly ignored')
chk(41 not in extract_principle_numbers('#41 is not a valid principle'),
    '#41 out-of-range correctly ignored')

# Multiple principles in one string
multi = extract_principle_numbers('Principles #1, #3, and #35 are key')
chk(multi == {1, 3, 35},
    f'Multiple #N in one string: expected {{1,3,35}} got {multi}')

In [ ]:
# ── Assertion suite — contradiction_accuracy ──────────────────────────────
print('--- contradiction_accuracy ---')

# From plan (lines 93-95): should NOT trigger on short generic words
ref_cp_false = {
    'improving_parameter': 'Ease of operation (#33)',
    'worsening_parameter':  'Device complexity (#36)',
}
chk(
    not contradiction_accuracy('the device operates in a complex field', ref_cp_false),
    'False-positive guard: generic sentence does NOT match (#33)/(#36)'
)

# Happy path: exact parameter IDs present
chk(
    contradiction_accuracy(
        'Improving ease of operation (#33) while reducing device complexity (#36)',
        ref_cp_false
    ),
    'Happy path: both parameter IDs (#33, #36) correctly detected'
)

# Fallback (no ID in param string): 5-char content word match
ref_cp_words = {
    'improving_parameter': 'structural strength',
    'worsening_parameter':  'device weight',
}
chk(
    contradiction_accuracy(
        'We improve structural strength at the cost of increased device weight',
        ref_cp_words
    ),
    'Fallback word match: "structural strength" + "device weight" correctly detected'
)

# Fallback false-positive: 3-char words only — should NOT match
ref_cp_short = {
    'improving_parameter': 'Use',
    'worsening_parameter':  'Age',
}
chk(
    not contradiction_accuracy('anything goes here', ref_cp_short),
    'Empty content words (len<5): correctly returns False'
)

# ── principle_accuracy ────────────────────────────────────────────────────
print('--- principle_accuracy ---')

chk(
    principle_accuracy(
        'Apply #3 Local quality and #35 parameter changes to the design.',
        ['#3 Local quality', '#35 Parameter Changes', '#10 Preliminary action']
    ),
    'principle_accuracy: 2 of 3 correct principles → True'
)
chk(
    not principle_accuracy(
        'Use segmentation to address the issue.',
        ['#35 Parameter Changes', '#40 Composite materials']
    ),
    'principle_accuracy: only 1 correct (segmentation=#1, not in ref) → False'
)
chk(
    principle_accuracy(
        'segmentation and feedback are key techniques.',
        ['#1 Segmentation', '#23 Feedback']
    ),
    'principle_accuracy: 2/2 named aliases → True'
)

print()

In [ ]:
# ── Final assertion summary ───────────────────────────────────────────────
total_checks = 10 + 7  # from both cells above (estimate; failures list is cumulative)
if failures:
    print(f'METRIC ASSERTIONS: {len(failures)} FAILED')
    for f in failures:
        print(f'  ✗ {f}')
else:
    print('ALL METRIC ASSERTIONS PASSED')

metrics_ok = len(failures) == 0

---

## Bölüm 3 — Özet Rapor

In [ ]:
# ── Özet rapor ────────────────────────────────────────────────────────────
n_pairs = len(dpo_pairs)
from collections import Counter as _Counter

TEACHER_THRESHOLD = 1500
VIABLE_MINIMUM    = 200

print('=' * 60)
print('       DPO DRY-RUN SUMMARY')
print('=' * 60)

# ── Bölüm 1: pair count ───────────────────────────────────────────────────
print(f'\n[Bölüm 1] DPO Pairs')
print(f'  Total valid pairs:        {n_pairs:,}')
print(f'  Source breakdown:         {dict(_Counter(p.get("_source", "?") for p in dpo_pairs))}')
print(f'  Unmatched (no chosen):    {len(unmatched):,}')

# ── Identical pair status ─────────────────────────────────────────────────
if IDENTICAL_BUG_FOUND:
    print(f'\n  [BUG REMAINS] Smart filter did not eliminate all identical pairs.')
    print( '  INVESTIGATE: judged_pass_by_id may not cover all cases.')
    print( '  nb02 will crash on Colab — review cell-5 smart filter logic.')
else:
    print(f'\n  [BUG FIXED] Smart filter eliminated all identical pairs. nb02 is safe for Colab.')

if n_pairs < VIABLE_MINIMUM:
    print(f'\n  STATUS: ALARM — only {n_pairs} pairs, below minimum of {VIABLE_MINIMUM}.')
    print( '          DPO training with this few pairs will likely overfit or produce noise.')
    print( '          ACTION NEEDED: Check judged.jsonl for PASS records with parent_seed_id.')
elif n_pairs < TEACHER_THRESHOLD:
    need = TEACHER_THRESHOLD - n_pairs
    print(f'\n  STATUS: Below teacher threshold — {need} more pairs needed to reach {TEACHER_THRESHOLD}.')
    print( '          Teacher API (DeepSeek-R1-70B via DeepInfra) WILL be triggered on Colab.')
    print( '          Estimated API cost: ~$0.02-0.05 per pair @ DeepInfra pricing.')
else:
    print(f'\n  STATUS: OK — {n_pairs} pairs >= teacher threshold ({TEACHER_THRESHOLD}).')
    print( '          Teacher API will NOT be triggered. Proceeding to DPO training is safe.')

# ── Bölüm 2: metric assertions ────────────────────────────────────────────
print(f'\n[Bölüm 2] Metric Assertions')
if metrics_ok:
    print('  STATUS: OK — all assertions passed.')
    print('  extract_principle_numbers, principle_accuracy, contradiction_accuracy')
    print('  behave correctly with the implemented fixes.')
else:
    print(f'  STATUS: FAIL — {len(failures)} assertion(s) failed.')
    print('  ACTION NEEDED: Review nb03 cell-6 helper implementations before running on Colab.')

# ── Action items ──────────────────────────────────────────────────────────
print(f'\n[Action Items Before Colab]')
action_n = 1
if IDENTICAL_BUG_FOUND:
    print(f'  {action_n}. INVESTIGATE smart filter — identical pairs still slipping through.')
    action_n += 1
if n_pairs < VIABLE_MINIMUM:
    print(f'  {action_n}. INVESTIGATE: too few pairs — check parent_seed_id coverage in judged.jsonl.')
    action_n += 1
elif n_pairs < TEACHER_THRESHOLD:
    print(f'  {action_n}. SET DEEPINFRA_API_KEY in Colab secrets before running notebook 02.')
    action_n += 1
if not metrics_ok:
    print(f'  {action_n}. FIX nb03 cell-6 metric helpers (see assertion failures above).')
    action_n += 1
if action_n == 1:
    print('  None — ready for Colab. Upload data to Drive and run 00 → 01 → 02 → 03.')

print('=' * 60)